# Optimal Clusterings

## Preliminaries

In [141]:
# Imports

# General
import numpy as np
import pandas as pd
import json
import itertools
import time
from IPython.display import display
import random
from joblib import Memory
from datetime import datetime
import csv

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe' # For plotly graphs to render in this environment

# Geography
import geopandas as gpd
from shapely.geometry import Point, Polygon


# NLP
import re
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
import nltk, pprint
# pprint.pp(nltk.data.path)
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet as wn
from nltk.corpus import names
import pycountry

# sk-learn modeling
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, Normalizer, MinMaxScaler
from sklearn.model_selection import ParameterGrid
from sklearn.decomposition import TruncatedSVD
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.base import clone
from sklearn.metrics import (
        silhouette_score,
        silhouette_samples,
        davies_bouldin_score, 
        calinski_harabasz_score)

import hdbscan
from hdbscan import prediction as hdb_pred

In [142]:
# Install extras for nltk (run once)


# nltk.download("averaged_perceptron_tagger")
# nltk.download('averaged_perceptron_tagger_eng')
# nltk.download('wordnet')
# nltk.download('omw-1.4')  # extra lemmas
# nltk.download("names") # List of names

In [143]:
# Set directory

PATH = "C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA"

In [144]:
# Load builds data

builds_df = pd.read_csv(f"{PATH}/PERMITS/builds.csv")

cluster_builds_df = model_ready(builds_df)

# examine_df('builds dataframe',builds_df)

In [145]:
# Load builds data 

demos_df = pd.read_csv(f"{PATH}/PERMITS/demos.csv")

cluster_demos_df = model_ready(demos_df)

# examine_df('demos dataframe',demos_df)

In [146]:
# Load builds data 

renos_df = pd.read_csv(f"{PATH}/PERMITS/renos.csv")

cluster_renos_df = model_ready(renos_df)

# examine_df('renos dataframe',renos_df)

## Define helper functions

In [147]:
# Define function to examine dataframes

def examine_df(name,df,
               include_stats = True,
               include_sample = True):
    
    """
    Check basic info about a dataframe df
    """
    
    print(f"\n\nNumber of records in the {name} is: {len(df)}\n")
    print(f"\n\nNumber of features in the {name} is: {len(df.columns)}\n")
    print(f"The columns in the {name} are: {df.columns}\n")
    print(f"\n Other info about {name}:\n")
    display(df.info())
    if include_stats == True:
        print(f'\n Basic statistical info about {name}:\n')
        display(df.describe())
    if include_sample == True:
        print(f"\n\nSample of records in the {name}:")
        display(df.head(5))

In [148]:
# Define functions to detect binary columns vs. continuous functions

def is_binary(series: pd.Series) -> bool:

    '''
    Detect binary columns in series
    '''
    
    vals = series.dropna().unique()
    return set(vals).issubset({0, 1}) and len(vals) <= 2

def select_binary_cols(X):

    '''
    Select subset of binary columns
    '''
    
    return [c for c in X.columns
            if c != "project_description"
            and is_binary(X[c])]

def select_cont_cols(X):

    '''
    Select subset of continuous columns
    '''
    
    cols = []
    for c in X.columns:
        if c == "project_description":
            continue
        s = X[c]
        # numeric dtype
        if np.issubdtype(s.dtype, np.number):
            if not is_binary(s):
                cols.append(c)
    return cols

In [149]:
# Define function to drop trivial columns from dataframe

def drop_trivial(df):

    """
    Drop all trivial (all 1 or all 0) columns from a dataframe
    """

    n_rows = len(df)
    
    to_drop = []

    for col in df.columns:
        if is_binary(df[col]) and (df[col].sum() == 0 or df[col].sum() == n_rows):
            to_drop.append(col)

    df = df.drop(columns = to_drop)
    return df

In [150]:
# Define function to make permit sub-dataframe model ready

def model_ready(df):

    """
    Drop columns that won't be used in modeling
    """

    return df.drop(columns = ['issue_date', 'geom','nbhd','zone'])

## Cluster Evaluation Tools

In [151]:
# Define function to extract chosen tokens from pipeline

def extract_tokens(pipe, print_output = True, n = 100):
    
    """
    Extract tokens from fitted pipeline
    """
    
    # Extract the fitted vectorizer from inside the pipeline
    fitted_vectorizer = pipe.named_steps["prep"] \
                            .named_transformers_["text"] \
                            .named_steps["tfidf"]
    
    # Vocabulary as dict {token: index}
    vocab_dict = fitted_vectorizer.vocabulary_
    
    # Reverse mapping: index -> token
    index_to_token = {i: t for t, i in vocab_dict.items()}
    
    # Sorted list of tokens by index
    tokens = [index_to_token[i] for i in range(len(index_to_token))]

    if print_output:
        print("Number of tokens:", len(tokens))
        print(f"\n\nList of {n} random tokens:\n\n", random.sample(tokens,n))

    return tokens, vocab_dict, fitted_vectorizer

In [152]:
# Define function to check influential tokens for SVD components

def svd_top_tokens(pipe, n_components = 5):

    """
    Extract top tokens for each SVD component from a fitted clustering pipeline
    """
    
    # Pull fitted SVD and TF-IDF from pipeline
    text_branch = (pipe.named_steps["prep"]
                        .named_transformers_["text"])
    svd = text_branch.named_steps["svd"]
    tfidf = text_branch.named_steps["tfidf"]

    # Build index->token list aligned to svd.components_
    # Use vocabulary_ to guarantee same ordering
    vocab = tfidf.vocabulary_                  # {token: index}
    idx_to_tok = {i: t for t, i in vocab.items()}
    terms = [idx_to_tok[i] for i in range(len(idx_to_tok))]

    comps = svd.components_
    comps_sample = comps[:n_components]

    print(f"Number of SVD components is: {len(comps)}")
    print(f"\n\nInfluential tokens for first {n_components} components are as follows:\n")

    for i, comp in enumerate(comps_sample):
        top_idx = np.argsort(comp)[-10:][::-1]    # Top 10 tokens
        top_terms = [terms[j] for j in top_idx]
        print(f"Component {i}: {', '.join(top_terms)}")

In [153]:
# Define function to evaluate clustering in terms of overall metrics


def overall_cluster_metrics(pipe, df, 
                            gmm = False,
                   print_output = True):
    """
    Compute and return series summarizing overall evaluation metrics 
    for a clustering pipeline. If gmm=True, also include AIC/BIC/etc.
    """

    prep = pipe.named_steps["prep"]
    est  = list(pipe.named_steps.values())[-1]  # last step (kmeans,gmm,hdbscan,...)

    # Transform to metric space used for clustering
    X = prep.transform(df)

    # Get labels: use .labels_ if present (e.g., KMeans/HDBSCAN), else .predict(X) (e.g., GMM)
    if hasattr(est, "labels_"):
        labels = est.labels_
    else:
        labels = est.predict(X)

    # Detect HDBSCAN 
    is_hdbscan = est.__class__.__name__.lower() == "hdbscan"

    # --- Silhouette score ---
    # For HDBSCAN: exclude noise (-1); return NaN if <2 clusters remain or too few samples
    if is_hdbscan:
        non_noise_mask = labels != -1
        if non_noise_mask.sum() >= 2 and len(np.unique(labels[non_noise_mask])) >= 2:
            sil = silhouette_score(X[non_noise_mask], labels[non_noise_mask], metric="euclidean")
        else:
            sil = np.nan
    else:
        # For other clusterers: compute on all points (as before)
        # Guard: if only one label, silhouette is undefined
        uniq = np.unique(labels)
        if len(uniq) >= 2:
            sil = silhouette_score(X, labels, metric="euclidean")
        else:
            sil = np.nan

    # --- Other indices (unchanged) ---
    ch = calinski_harabasz_score(X, labels) if len(np.unique(labels)) >= 2 else np.nan
    db = davies_bouldin_score(X, labels)    if len(np.unique(labels)) >= 2 else np.nan

    results = pd.Series({
        "Inertia Score (lower better)": float(est.inertia_) if hasattr(est, "inertia_") else np.nan,
        "Silhouette Score (higher better)": sil,
        "Calinski Harabasz Index (higher better)": ch,
        "Davies Bouldin Index (lower better)": db,
    })

    # Noise metric for HDBSCAN
    if is_hdbscan:
        noise_frac = float((labels == -1).mean())
        n_clusters_no_noise = int(len(np.unique(labels[labels != -1]))) if np.any(labels != -1) else 0
        results["Noise Fraction (density methods)"] = noise_frac
        results["#Clusters (excl. noise)"] = n_clusters_no_noise

    # Extra scores for GMM
    if gmm:
        # Some GMM implementations require array-like; X is fine here
        results["AIC (lower better)"] = est.aic(X)
        results["BIC (lower better)"] = est.bic(X)

    if print_output:
        try:
            display(results)
        except NameError:
            print(results)

    return results


In [154]:
# Define function to compute basic info about each cluster of a fitted pipeline

def cluster_averages(pipe, df):
    
    """
    Provide per-cluster summary using original pipe
    """
    
    # Get estimator
    est  = list(pipe.named_steps.values())[-1]

    df = df.copy()
    value_col = "project_value"
    binary_cols = [c for c in df.columns if c != value_col and is_binary(df[c])]

    # Transform to pipeline feature space (for silhouette)
    prep = pipe.named_steps["prep"]
    prep_df = prep.transform(df)

    # Labels: use .labels_ if present (e.g., KMeans); else .predict (e.g., GMM)
    labels = getattr(est, "labels_", None)
    if labels is None:
        labels = est.predict(prep_df)

        # Detect HDBSCAN and drop noise if present
    est_name = est.__class__.__name__.lower()
    is_hdbscan = ("hdbscan" in est_name)

    if is_hdbscan:
        non_noise_mask = (labels != -1)
        # If everything is noise, return an empty (but well-formed) table
        if non_noise_mask.sum() == 0:
            return pd.DataFrame(columns=[
                "cluster","size","share",f"{value_col}_mean","silhouette", *binary_cols
            ])
        # Filter to non-noise for all downstream calcs
        df = df.loc[non_noise_mask].copy()
        if hasattr(prep_df, "toarray"):  # handles sparse
            prep_df = prep_df[non_noise_mask]
        else:
            prep_df = prep_df[non_noise_mask]
        labels = labels[non_noise_mask]
    
    # Group & aggregate on original features
    g = df.groupby(labels)
    out = pd.DataFrame({
        "cluster": g.size().index.astype(int),
        "size": g.size().values,
    }).sort_values("cluster")

    out["share"] = out["size"] / len(df)
    out[f"{value_col}_mean"] = g[value_col].mean().values

    for c in binary_cols:
        out[c] = g[c].mean().values

    # Per-cluster silhouette mean (only if at least 2 clusters)
    try:
        if len(np.unique(labels)) >= 2:
            sil_samp = silhouette_samples(prep_df, labels, metric="euclidean")
            sil_means = pd.Series(sil_samp).groupby(labels).mean()
            out["silhouette"] = out["cluster"].map(sil_means.to_dict()).astype(float)
        else:
            out["silhouette"] = np.nan
    except Exception:
        out["silhouette"] = np.nan

    # pretty formatting
    pretty = out.copy()
    pretty["share"] = pretty["share"].map(lambda x: f"{x:.1%}")
    pretty[f"{value_col}_mean"] = pretty[f"{value_col}_mean"].map(lambda x: f"${x:,.0f}")
    pretty["silhouette"] = pretty["silhouette"].map(lambda x: f"{x:.3f}" if pd.notnull(x) else "")

    return pretty.reset_index(drop=True)

In [155]:
# Define function to determine top ten tokens for each cluster

def top_tokens_per_cluster(pipe, df, top_k=10, print_output=True):
    
    """
    For each cluster, return the top-k tokens by frequency
    """
    
    # Get estimator
    est  = list(pipe.named_steps.values())[-1]

    # Labels: use .labels_ if present; else predict on transformed features
    prep = pipe.named_steps["prep"]
    Xt = prep.transform(df)
    labels = getattr(est, "labels_", None)
    if labels is None:
        labels = est.predict(Xt)

    tokens, vocab, tfidf = extract_tokens(pipe, print_output=False)

    count_vec = CountVectorizer(
        tokenizer=tfidf.tokenizer,
        token_pattern=None,
        lowercase=False,
        vocabulary=vocab,
        ngram_range=tfidf.ngram_range,
    )

    counts = count_vec.transform(df["project_description"])

    out_rows = []
    for cid in np.unique(labels):
        idx = (labels == cid)
        n_docs = idx.sum()

        cluster_sum = np.asarray(counts[idx].sum(axis=0)).ravel()
        cluster_sum = cluster_sum / max(n_docs, 1)

        if top_k < len(cluster_sum):
            top_idx = np.argpartition(cluster_sum, -top_k)[-top_k:]
        else:
            top_idx = np.arange(len(cluster_sum))

        top_idx = top_idx[np.argsort(cluster_sum[top_idx])[::-1]]

        for rank, j in enumerate(top_idx[:top_k], start=1):
            out_rows.append({
                "cluster": int(cid),
                "token": tokens[j],
                "score": float(cluster_sum[j]),
                "rank": rank
            })

    result = (pd.DataFrame(out_rows)
                .sort_values(["cluster", "rank"])
                .reset_index(drop=True))

    if print_output:
        for cid in np.unique(labels):
            sub = result[result["cluster"] == cid][["token", "score", "rank"]]
            print(f"\nTop {top_k} tokens for cluster {int(cid)}")
            print(sub.to_string(index=False))

    return result

In [156]:
# Define function to extract 3D representation of clusters as a dataframe


def cluster_embedding_3d(pipe, df):
    
    """
    Returns a compact DataFrame with:
      - 'project_value' (z-axis)
      - 'nlp_x'         (SVD dim 1 of text branch)
      - 'nlp_y'         (SVD dim 2 of text branch)
      - 'cluster'       (labels from final estimator)
    """
    
    # Final estimator (last step of pipeline)
    est = pipe.steps[-1][1]

    # Preprocessor
    prep = pipe.named_steps["prep"]

    # Transform once (useful if we ever need it)
    _ = prep.transform(df)

    # Labels: use .labels_ if available; otherwise try predict()
    labels = getattr(est, "labels_", None)
    if labels is None and hasattr(est, "predict"):
        labels = est.predict(_)
    if labels is None:
        raise ValueError("Could not obtain cluster labels from the final estimator.")

    # Z-axis
    project_value = df["project_value"].astype(float).to_numpy()

    # Text 2D from fitted text branch
    text_branch = prep.named_transformers_["text"]
    tfidf = text_branch.named_steps["tfidf"]
    svd_text = text_branch.named_steps["svd"]

    Z_text = svd_text.transform(tfidf.transform(df["project_description"]))
    nlp_x = Z_text[:, 0]
    if getattr(svd_text, "n_components", 1) >= 2 and Z_text.shape[1] >= 2:
        nlp_y = Z_text[:, 1]
    else:
        # Fallback: duplicate first dim if SVD has only 1 component
        nlp_y = nlp_x.copy()

    out = pd.DataFrame({
        "project_value": project_value,
        "nlp_x": nlp_x,
        "nlp_y": nlp_y,
        "cluster": np.asarray(labels, dtype=int)
    })

    return out

In [157]:
# Define function to generate 3D scatter plot of clusters

def plot_clusters_3d(pipe, df):
    
    """
    Builds 3D embedding and renders a Plotly 3D scatter:
      x = nlp_x (text SVD dim 1)
      y = nlp_y (text SVD dim 2)
      z = project_value (log-scaled)
      color = cluster
    """
    
    emb = cluster_embedding_3d(pipe, df)

    fig = px.scatter_3d(
        emb,
        x="nlp_x",
        y="nlp_y",
        z="project_value",
        color="cluster",
        hover_data={
            "project_value": ":,.0f",
            "nlp_x": ":.3f",
            "nlp_y": ":.3f",
            "cluster": True
        },
        title="3D Cluster Visualization (Text SVD 2D • Project Value [log scale])",
        opacity=0.8
    )

    # Make points small and clear
    fig.update_traces(marker=dict(size=4))

    # Axis labels + log scale on z
    fig.update_layout(
        scene=dict(
            xaxis_title="Text SVD — dim 1",
            yaxis_title="Text SVD — dim 2",
            zaxis=dict(
                title="Project Value (log scale)",
                type="log"
            )
        ),
        legend_title_text="Cluster"
    )

    return fig

In [158]:
# Define function that can run all cluster evaluation tools consecutively

def full_evaluator(pipe, df,
                   ext_tokens = True,
                   svd_tokens = True,
                   overall_metrics = True,
                   cluster_avgs = True,
                   cluster_tokens = True,
                    visualize = True):

    """
    Run all (or a subset) of the different cluster evaluation functions above
    """

    # Initialize dictionary to store runtims
    runtimes = {}

    full_start = time.time()

    if ext_tokens:
        print("\n\nExtracting all tokens from tf-idf vectorizer...\n\n")
        t0 = time.time()
        extract_tokens(pipe)
        runtimes["Extracting all tokens"] = (time.time() - t0) / 60

    if svd_tokens:
        print("\n\nExtracting top tokens for SVD components...\n\n")
        t0 = time.time()
        svd_top_tokens(pipe)
        runtimes["Top tokens per SVD component"] = (time.time() - t0) / 60

    if overall_metrics:
        print("\n\nComputing overall cluster metrics...\n\n")
        t0 = time.time()
        metric_series = overall_cluster_metrics(pipe, df, print_output = False)
        print("Our model's performance metrics are as follows:\n\n")
        display(metric_series)
        runtimes["Overall metrics"] = (time.time() - t0) / 60

    if cluster_avgs:
        print("\n\nComputing feature averages by cluster...\n\n")
        t0 = time.time()
        cluster_avgs_df = cluster_averages(pipe, df)
        print("Our clusters have the following feature means:\n\n")
        print(cluster_avgs_df)
        runtimes["Cluster averages"] = (time.time() - t0) / 60

    if cluster_tokens:
        print("\n\nComputing top ten tokens for each cluster...\n\n")
        t0 = time.time()
        top_tokens_per_cluster(pipe, df)
        runtimes["Top tokens per cluster"] = (time.time() - t0) / 60

    if visualize:
        print("\n\n Preparing 3D clustering visualization...\n\n")
        t0 = time.time()
        fig = plot_clusters_3d(pipe, df)
        fig.show()
        runtimes["3D visualization"] = (time.time() - t0) / 60

    full_elapsed_min = (time.time() - full_start) / 60

    print("\n\n=== Step runtimes (minutes) ===")
    for name, minutes in runtimes.items():
        print(f"- {name}: {minutes:.2f} min")

    print(f"\nTotal runtime: {full_elapsed_min:.2f} minutes.")

## Define Text Processing Pipeline

In [159]:
# Define lemmatizer

lemmatizer = WordNetLemmatizer()

def _to_wordnet_pos(penn_tag):

    '''
    Map Penn Treebank tags to WordNet POS for better lemmatization
    '''
    
    if penn_tag.startswith('J'): return wn.ADJ
    if penn_tag.startswith('V'): return wn.VERB
    if penn_tag.startswith('N'): return wn.NOUN
    if penn_tag.startswith('R'): return wn.ADV
    return wn.NOUN  # default

In [160]:
# Build lists of stopwords

# Name stopwords
NAME_STOPWORDS = set(n.lower() for n in names.words())

# Canadian provinces and territories
CANADIAN_REGIONS = {p.name.lower() for p in pycountry.subdivisions.get(country_code="CA")}

# Manually curated set of proper nouns / non-informative tokens to exclude
CUSTOM_STOPWORDS = {
    # Directions / locations
    "east", "west", "north", "south", "avenue", "ave", "street", "road", "lane",
    "rear","upper","lower",

    # Geography
    "vancouver", "burnaby", "downtown", "richmond", "kitsilano", "kerrisdale", "oakridge",
    "surrey","coquitlam","new westminster",
    "delta","north vancouver","west vancouver","port moody","maple ridge",
    "langley","abbotsford","victoria","kelowna","kamloops","nanaimo",
    "toronto","montreal","calgary","edmonton","ottawa","winnipeg","hamilton",

    # Common personal names (from your token list)
    "alan", "peng", "wang", "kumar", "antony", "alam", "biswas", "chen", "fong", 
    "guan", "guo", "henry", "hsu", "hui", "jian", "jason", "jeff", "peter", 
    "prahalad", "shahidul", "shambhu", "sharma", "sidhu", "shikder", "yatendra", 
    "zhao", 

    # bureaucracy / boilerplate
    "accordance","pursuant","shall","permit","permits","permitted",
    "application","applicant","approve","approved","approval","issue",
    "issued","issuance","plan","plans","planning","schedule","schedules",
    "document","file","reference","regard","response","result","results",
    "review","revision","revise","status","submit","submitted","submission",
    "prior","obtain","obtained","process","validate","value",

    # vague verbs/adverbs that don’t help clustering
    "provide","provided","provides","propose","proposed","proposes",
    "include","includes","including","require","requires","required",
    "requirement","requirements","achieve","approximately","fully","future",
    "completion","complete","completed","comply","conformance","conform",
    "satisfaction","feature","features","performance",

    # standards / codes and recurring abbreviations
    "vbbl","nfpa","aama","nafs","csa101","hpo","tbc","thpo","tschedule",
    "tcovenant","tno","tentire","texternal","tb1","tbar","thrv",

    # generic high-freq nouns
    "area","site","component","equipment","note","notes",
    "work","works","year","years","dwell","dwells","dwelling","dwellings"

    # Misc project‐specific junk that isn’t helpful
    "aibc", "aama", "nafs", "csa101", "vbbl", "nfpa", "hpo", "tbc", "thpo",
    "tschedule", "tcovenant", "tno", "tentire", "texternal", "tb1", "tbar",
    "thrv", "pandemic", "eng",
}

In [161]:
# Custom normalizer for spelling/lemmatization

NORMALIZE_MAP = {
    "instal": "install",
    "story": "storey",      # unify to Canadian usage
    "storeys": "storey",    # plural to canonical (lemmatizer also helps)
}

def normalize_spelling(tok: str) -> str:

    """
    Normalize spelling and lemmatize for custom tokens
    """
    
    t = tok
    # map common variants
    t = NORMALIZE_MAP.get(t, t)
    # collapse obvious duplicates like "basement basement"
    t = re.sub(r"(.)\1{2,}", r"\1", t)   # brutal de-noise for typos (e.g., ttthe->the)
    return t

In [162]:
# Define custom tokenizer

def clean_tokenizer(text):
    
    """
    Clean and tokenize project descriptions:
    - remove proper nouns (NNP, NNPS)
    - lowercase
    - keep words + numbers
    - remove stopwords & very short tokens
    - lemmatize remaining tokens
    """

    # keep only letters, digits, whitespace
    text = re.sub(r"[^A-Za-z0-9\s]", " ", text)
    tokens = text.split()

    # POS tagging: remove proper nouns (NNP, NNPS) before lowercasing
    pos_tags = nltk.pos_tag(tokens)
    tokens = [t for t, pos in pos_tags if pos not in ("NNP", "NNPS")]

    # lowercase
    tokens = [t.lower() for t in tokens]

    # normalize spelling / variants
    tokens = [normalize_spelling(t) for t in tokens]

    # filter stopwords, names, Canadian geography, custom junk
    tokens = [
        t for t in tokens
        if t not in ENGLISH_STOP_WORDS
        and t not in NAME_STOPWORDS
        and t not in CANADIAN_REGIONS
        and t not in CUSTOM_STOPWORDS
        and len(t) > 2
        and not re.match(r"^\d", t)   # remove numbers or digit-prefixed tokens
    ]

    # Tag tokens
    pos_tags2 = nltk.pos_tag(tokens)

    # lemmatize
    tokens = [lemmatizer.lemmatize(t, _to_wordnet_pos(pos)) for t, pos in pos_tags2]

    return tokens

In [163]:
# Define vectorizer 

vectorizer = TfidfVectorizer(
    tokenizer=clean_tokenizer,  # Use your custom tokenizer function
    token_pattern=None,     # <-- important when using custom tokenizer
    ngram_range=(1,2),
    max_features=1000,           # Keep the top tokens
    min_df=10,                   # Minimum document frequency of 10
    max_df=0.8,            # filter out too common terms
    lowercase=False       # tokenizer already lowercases
)


In [164]:
# Define text processing pipeline

text_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        tokenizer=clean_tokenizer,
        token_pattern=None,        # Important with custom tokenizer
        ngram_range=(1, 2),
        max_features=3000,
        min_df=10,
        max_df=0.8,
        lowercase=False            # Tokenizer already lowercases
    )),
    ("svd", TruncatedSVD(n_components=5, random_state=42)),
    ("scale", StandardScaler())    # scale dense SVD components for KMeans
])   

In [165]:
# Feature processor (text processing and scaling)

preprocess = ColumnTransformer(
    transformers=[
        ("text", text_pipe, "project_description"),   # text branch
        ("scale_cont", StandardScaler(), select_cont_cols), # continuous branch
        ("pass_bin",  "passthrough",   select_binary_cols) # binary flags untouched
    ],
    remainder="drop"
)

## Clustering with K-Means

### K-Means Pipeline Set Up

In [50]:
# Define K-Means clustering pipeline

kmeans = KMeans(
    n_clusters = 4,  
          init = "k-means++",
        n_init = "auto",   
      max_iter = 300,
           tol = 1e-4,
       verbose = 0,
  random_state = 42,
    algorithm="lloyd", 
)

cluster_pipeline = Pipeline([
    ("prep", preprocess),
    ("kmeans", kmeans),
])

In [60]:
# Function to run full K-Means pipeline

def run_kmeans_pipeline(df, n_clusters=9, svd_n_components=5, tfidf_max_features=1000):
    
    """
    Run kmeans pipeline with chosen parameters
    """

    start = time.time()
    
    # Update params inside the pipeline
    cluster_pipeline.set_params(
        kmeans__n_clusters=n_clusters,
        prep__text__svd__n_components=svd_n_components,
        prep__text__tfidf__max_features=tfidf_max_features
    )
    
    # Fit the pipeline
    fitted = cluster_pipeline.fit(df)

    end = time.time()

    runtime = (end - start)/60

    print(f"K-Means pipeline fitted in {runtime:.2f} minutes.")
    
    return fitted

### Optimal Clusterings: Builds

In [78]:
cluster_builds_df['project_value'].describe()

count    9.464000e+03
mean     2.130362e+06
std      1.512969e+07
min      1.019777e+03
25%      2.388613e+05
50%      7.070169e+05
75%      9.785543e+05
max      8.408412e+08
Name: project_value, dtype: float64

In [80]:
# Drop extreme outliers

print(f"Number of records in our model-ready builds dataframe is {len(cluster_builds_df)}.")

cluster_builds_df = cluster_builds_df[cluster_builds_df['project_value']<500000000]

print(f"After dropping extreme outliers, number of records in our model-ready builds dataframe is {len(cluster_builds_df)}.")

Number of records in our model-ready builds dataframe is 9464.
After dropping extreme outliers, number of records in our model-ready builds dataframe is 9462.


In [85]:
# Run K-Means pipeline with optimal parameters (by sil. score)

builds_kmeans_fitted_00 = run_kmeans_pipeline(cluster_builds_df, n_clusters=8, svd_n_components=5, tfidf_max_features=1000)

K-Means pipeline fitted in 2.16 minutes.


In [86]:
# Run full evaluator

full_evaluator(builds_kmeans_fitted_00, cluster_builds_df)



Extracting all tokens from tf-idf vectorizer...


Number of tokens: 1000


List of 100 random tokens:

 ['sink', 'low develop', 'address retained', 'asterisk building', 'small', 'bylaw effective', 'underground park', 'garage provide', 'space provide', 'floor building', 'locate second', 'space have', 'build authorized', 'sprinklered registered', 'access access', 'land', 'district', 'indicate', 'pump', 'sprinklered unit', 'second', 'stamp lieu', 'construction', 'pump locate', 'duplex secondary', 'certify construct', 'stairs', 'thermal wall', 'unit sink', 'technical', 'residential project', 'room structural', 'relate', 'obtain protection', 'tstructural geotechnical', 'entire', 'structural geotechnical', 'building cellar', 'mini', 'park provide', 'roof deck', 'new mixed', 'access locate', 'house assign', 'underground', 'project sprinkler', 'common underground', 'accommodate improve', 'construct cellar', 'attach car', 'access maintain', 'register stratum', 'entire building', 'associate', 

Inertia Score (lower better)               19215.427686
Silhouette Score (higher better)               0.373481
Calinski Harabasz Index (higher better)     3385.156560
Davies Bouldin Index (lower better)            0.941098
dtype: float64



Computing feature averages by cluster...


Our clusters have the following feature means:


   cluster  size  share project_value_mean  duplex_w_secondary_suite  \
0        0  1442  15.2%           $888,993                  0.217060   
1        1  1655  17.5%           $702,084                  0.046526   
2        2   873   9.2%           $241,283                  0.000000   
3        3  2048  21.6%           $995,537                  0.079590   
4        4  1240  13.1%         $5,669,004                  0.006452   
5        5    70   0.7%        $90,261,834                  0.000000   
6        6  1586  16.8%           $230,763                  0.000000   
7        7   548   5.8%           $246,122                  0.000000   

   laneway_house    duplex  multiple_conversion_dwelling  dwelling_unit  \
0       0.000693  0.314147                      0.000000       0.000000   
1       0.375227  0.084592                      0.000000       0.003625   
2       0.991982  0.000000      



=== Step runtimes (minutes) ===
- Extracting all tokens: 0.00 min
- Top tokens per SVD component: 0.00 min
- Overall metrics: 1.90 min
- Cluster averages: 1.89 min
- Top tokens per cluster: 3.72 min
- 3D visualization: 3.73 min

Total runtime: 11.23 minutes.


#### K-Means Clustering Results — New Building Permits

Below is a summary of the 8 clusters, their defining characteristics, and interpretations.

---

##### Cluster 0 — Suites & Duplexes
- **Size / Share:** 1,442 permits (15.2%)
- **Avg Project Value:** **$888,993**
- **Top Tokens:** suite, number, build, basement, floor
- **Features:** High rate of duplexes with secondary suites and some single-detached houses.
- **Interpretation:** Typical **moderate-value low-density housing** with secondary suites.

---

##### Cluster 1 — Small Detached & Conversions
- **Size / Share:** 1,655 permits (17.5%)
- **Avg Project Value:** **$702,084**
- **Top Tokens:** sprinklered, face, expose, conversion, garage
- **Features:** Mix of small single-detached houses and conversions, some laneway houses.
- **Interpretation:** **Lower-value detached homes**, often with fire-safety upgrades or small conversions.

---

##### Cluster 2 — Laneway Houses (Lower Value)
- **Size / Share:** 873 permits (9.2%)
- **Avg Project Value:** **$241,283**
- **Top Tokens:** principal, house, laneway, building survey
- **Features:** Nearly all laneway houses, very low prevalence of other dwelling types.
- **Interpretation:** **Small laneway projects** attached to existing houses, **low-value permits**.

---

##### Cluster 3 — Detached Homes with Secondary Suites
- **Size / Share:** 2,048 permits (21.6%)
- **Avg Project Value:** **$995,537**
- **Top Tokens:** unit, suite, secondary, basement, family
- **Features:** Dominated by single-detached houses, many with secondary suites.
- **Interpretation:** **Mainstream detached housing cluster**, mid-to-high value, families with suites.

---

##### Cluster 4 — Multi-Unit & Mixed Residential
- **Size / Share:** 1,240 permits (13.1%)
- **Avg Project Value:** **$5,669,004**
- **Top Tokens:** unit, park, build, garage, family
- **Features:** High share of multiple dwellings and single-detached houses, some conversions.
- **Interpretation:** **Larger multi-unit developments** (townhomes, condos) with much higher value.

---

##### Cluster 5 — Large Multi-Dwelling Complexes
- **Size / Share:** 70 permits (0.7%)
- **Avg Project Value:** **$90,261,834**
- **Top Tokens:** unit, level, stage, underground, certify
- **Features:** Almost entirely multiple dwellings, very high average value.
- **Interpretation:** **Major residential complexes** (e.g. high-rises), extremely high-value projects.

---

##### Cluster 6 — Laneway Houses (Very Low Value)
- **Size / Share:** 1,586 permits (16.8%)
- **Avg Project Value:** **$230,763**
- **Top Tokens:** house, laneway, law, park space, construct
- **Features:** Almost exclusively laneway houses, with zoning/legal references.
- **Interpretation:** **Small-scale laneway construction**, lowest-cost permits overall.

---

##### Cluster 7 — Suites, Numbers & Legal References
- **Size / Share:** 548 permits (5.8%)
- **Avg Project Value:** **$246,122**
- **Top Tokens:** number, suite, build, house, law, post
- **Features:** Strong token references to numbering, legal compliance, some laneway houses.
- **Interpretation:** **Administrative / small projects** involving suites or legal requirements, very low value.

---


### Optimal Clusterings: Renos

In [87]:
# Run K-Means pipeline with optimal parameters  (by sil. score)

renos_kmeans_fitted_00 = run_kmeans_pipeline(cluster_renos_df, n_clusters=6, svd_n_components=5, tfidf_max_features=500)

K-Means pipeline fitted in 0.93 minutes.


In [88]:
# Run full evaluator

full_evaluator(renos_kmeans_fitted_00, cluster_renos_df)



Extracting all tokens from tf-idf vectorizer...


Number of tokens: 500


List of 100 random tokens:

 ['residential building', 'remove wall', 'car', 'letter', 'roof exist', 'non loadbearing', 'install new', 'title', 'washroom', 'cover', 'mount', 'overhead', 'enclose balcony', 'alteration install', 'suite scope', 'interior wall', 'air', 'occupy', 'specifically', 'material', 'washer', 'use family', 'add new', 'identify', 'bar', 'drawing', 'load', 'load bear', 'pump', 'open concept', 'trigger', 'stratum', 'sprinklered', 'trigger choose', 'silence', 'storage room', 'hardwired', 'second', 'build scope', 'patio', 'frame', 'remove exist', 'visible building', 'enforcement', 'alteration replace', 'millwork', 'document', 'plumbing', 'removal', 'living', 'domestic', 'family building', 'concrete', 'unauthorized', 'mixed use', 'smoke alarm', 'suite exit', 'heat pump', 'ensuite', 'attach', 'laundry room', 'responsibility identify', 'change', 'damage repair', 'create open', 'scope removal', 'match

Inertia Score (lower better)               16695.000479
Silhouette Score (higher better)               0.383701
Calinski Harabasz Index (higher better)     3361.342169
Davies Bouldin Index (lower better)            0.790953
dtype: float64



Computing feature averages by cluster...


Our clusters have the following feature means:


   cluster  size  share project_value_mean  duplex_w_secondary_suite  \
0        0  2542  32.3%            $75,681                  0.000000   
1        1  3629  46.1%            $97,853                  0.000827   
2        2   810  10.3%            $57,172                  0.004938   
3        3   599   7.6%            $66,104                  0.000000   
4        4   254   3.2%            $76,848                  0.003937   
5        5    41   0.5%         $2,391,476                  0.000000   

   laneway_house    duplex  multiple_conversion_dwelling  dwelling_unit  \
0       0.000000  0.008655                      0.011015       0.042486   
1       0.007716  0.047672                      0.026178       0.028934   
2       0.000000  0.002469                      0.023457       0.004938   
3       0.000000  0.006678                      0.001669       0.255426   
4       0.007874  0.015748



=== Step runtimes (minutes) ===
- Extracting all tokens: 0.00 min
- Top tokens per SVD component: 0.00 min
- Overall metrics: 0.91 min
- Cluster averages: 0.94 min
- Top tokens per cluster: 1.74 min
- 3D visualization: 1.91 min

Total runtime: 5.50 minutes.


#### K-Means Clustering Results — Renovation Permits

Below is a summary of the 6 clusters, their defining characteristics, and interpretations.

---

##### Cluster 0 — Multi-Unit Renovations (Low Value)
- **Size / Share:** 2,542 permits (32.3%)
- **Avg Project Value:** **$75,681**
- **Top Tokens:** floor, unit, kitchen, multiple, bathroom
- **Features:** Predominantly multiple dwellings, often interior renovations (floors, kitchens, bathrooms).
- **Interpretation:** **Routine low-cost renovations** in existing multi-unit buildings.

---

##### Cluster 1 — Single-Detached Interior/Exterior Work
- **Size / Share:** 3,629 permits (46.1%)
- **Avg Project Value:** **$97,853**
- **Top Tokens:** building, new, floor, family, remove, interior, exterior, door
- **Features:** Largest cluster; dominated by single-detached houses with interior and exterior renovations.
- **Interpretation:** **Mainstream home renovations**, moderate in scope and value.

---

##### Cluster 2 — Secondary Suite Renovations
- **Size / Share:** 810 permits (10.3%)
- **Avg Project Value:** **$57,172**
- **Top Tokens:** suite, secondary, secondary suite, family, access, law
- **Features:** Nearly all projects involve secondary suites in detached houses.
- **Interpretation:** **Creation or modification of secondary suites**, lowest-value renovation cluster.

---

##### Cluster 3 — Multi-Unit Building Alterations
- **Size / Share:** 599 permits (7.6%)
- **Avg Project Value:** **$66,104**
- **Top Tokens:** unit, building, multiple, direct alteration, improvement
- **Features:** Heavy focus on multi-dwelling units and “alterations/improvements” language.
- **Interpretation:** **Alterations in existing apartment/condo buildings**, mid-low value.

---

##### Cluster 4 — Energy & System Upgrades
- **Size / Share:** 254 permits (3.2%)
- **Avg Project Value:** **$76,848**
- **Top Tokens:** stage, applicable, upgrade, support, document, energy, electrical, plumbing
- **Features:** Multi-dwellings with focus on upgrades (energy efficiency, systems, compliance).
- **Interpretation:** **Targeted system upgrades and retrofits**, moderate value but narrow scope.

---

##### Cluster 5 — Large-Scale Renovations
- **Size / Share:** 41 permits (0.5%)
- **Avg Project Value:** **$2,391,476**
- **Top Tokens:** new, building, roof, repair, replace, exterior
- **Features:** Multiple dwellings, large renovation scope including roofs, exteriors, and major repairs.
- **Interpretation:** **Major renovation projects** in large buildings, very high project values.

---


### Optimal Clusterings: Demos

#### First Optimal (with 10 clusters)

In [89]:
# Run K-Means pipeline with optimal parameters (by sil. score)

demos_kmeans_fitted_00 = run_kmeans_pipeline(cluster_demos_df, n_clusters=10, svd_n_components=8, tfidf_max_features=180)

K-Means pipeline fitted in 0.39 minutes.


In [90]:
# Run full evaluator

full_evaluator(demos_kmeans_fitted_00, cluster_demos_df)



Extracting all tokens from tf-idf vectorizer...


Number of tokens: 180


List of 100 random tokens:

 ['construction waste', 'single family', 'deconstruction confirm', 'green recycling', 'recycle non', 'number', 'receipt email', 'abatement', 'deconstruction recycle', 'associate green', 'recycle material', 'exist multiple', 'building hour', 'confirm', 'hour advance', 'phone number', 'family build', 'disposal', 'building update', 'conduct grade', 'recycle percent', 'shore', 'deconstruction exist', 'advance demolition', 'outside deconstruction', 'directly', 'energy', 'suspend', 'salvage', 'construction material', 'document', 'subdivision', 'recycling update', 'rezoning', 'suspend pending', 'percent', 'pending', 'pre', 'single', 'soil incidental', 'email', 'demolition call', 'demolish deconstruction', 'recycle hour', 'green update', 'residential', 'multiple building', 'hazardous', 'confirm acceptable', 'family relate', 'hour', 'account deconstruction', 'subject green', 'recycle deconstr

Inertia Score (lower better)               8535.688657
Silhouette Score (higher better)              0.607948
Calinski Harabasz Index (higher better)    3381.594476
Davies Bouldin Index (lower better)           0.659871
dtype: float64



Computing feature averages by cluster...


Our clusters have the following feature means:


   cluster  size  share project_value_mean  duplex_w_secondary_suite  \
0        0  1471  25.7%            $17,020                  0.028552   
1        1   975  17.1%            $28,873                  0.017436   
2        2   505   8.8%            $17,568                  0.023762   
3        3   914  16.0%            $17,378                  0.031729   
4        4   516   9.0%            $16,881                  0.017442   
5        5   375   6.6%            $70,820                  0.000000   
6        6   450   7.9%            $19,453                  0.002222   
7        7   315   5.5%            $43,507                  0.000000   
8        8   190   3.3%            $22,493                  0.000000   
9        9     3   0.1%         $4,648,644                  0.000000   

   laneway_house    duplex  multiple_conversion_dwelling  dwelling_unit  \
0       0.000680  0.043508            



=== Step runtimes (minutes) ===
- Extracting all tokens: 0.00 min
- Top tokens per SVD component: 0.00 min
- Overall metrics: 0.38 min
- Cluster averages: 0.33 min
- Top tokens per cluster: 0.72 min
- 3D visualization: 0.75 min

Total runtime: 2.18 minutes.


#### K-Means Clustering Results — Demolition / Deconstruction Permits

Below is a summary of the 10 clusters, their defining characteristics, and interpretations.

---

##### Cluster 0 — Standard Family House Demolitions
- **Size / Share:** 1,471 permits (25.7%)
- **Avg Project Value:** **$17,020**
- **Top Tokens:** family building, low demolish, construction
- **Features:** Mostly single-detached houses, some duplexes with suites.
- **Interpretation:** **Typical low-value demolitions** of small family houses.

---

##### Cluster 1 — Demolitions with Recycling / Contractors
- **Size / Share:** 975 permits (17.1%)
- **Avg Project Value:** **$28,873**
- **Top Tokens:** family building, low demolish, recycle, contractor, demo
- **Features:** Single-detached houses with demolition contractors and recycling involved.
- **Interpretation:** **Higher-value demolitions** emphasizing recycling and contractor oversight.

---

##### Cluster 2 — Green Deconstruction & Recycling
- **Size / Share:** 505 permits (8.8%)
- **Avg Project Value:** **$17,568**
- **Top Tokens:** deconstruction, family building, mean deconstruction, recycle, green
- **Features:** Single-detached houses, heavy emphasis on green building and recycling.
- **Interpretation:** **Deconstruction-focused projects** (salvage and recycling) at modest value.

---

##### Cluster 3 — Hazardous / Non-Hazardous Waste
- **Size / Share:** 914 permits (16.0%)
- **Avg Project Value:** **$17,378**
- **Top Tokens:** construction, non hazardous, hazardous, waste, recycle
- **Features:** Single-detached and duplex demolitions referencing hazardous materials.
- **Interpretation:** **Waste-handling demolitions** with environmental/hazard considerations.

---

##### Cluster 4 — Green & Energy Recycling
- **Size / Share:** 516 permits (9.0%)
- **Avg Project Value:** **$16,881**
- **Top Tokens:** green, recycling, subject, low demolish
- **Features:** Strong “green” and recycling themes, mostly detached houses.
- **Interpretation:** **Eco-focused demolitions**, environmentally compliant projects.

---

##### Cluster 5 — High-Value Demolitions (Large Projects)
- **Size / Share:** 375 permits (6.6%)
- **Avg Project Value:** **$70,820**
- **Top Tokens:** demolition, grade, outside, call, advance
- **Features:** Mix of duplexes and multiple dwellings, higher values.
- **Interpretation:** **Larger-scale demolitions**, more complex preparation/permits.

---

##### Cluster 6 — Bylaw & Energy Compliance
- **Size / Share:** 450 permits (7.9%)
- **Avg Project Value:** **$19,453**
- **Top Tokens:** bylaw effective, energy bylaw, update energy
- **Features:** Detached houses with strong references to bylaws and energy updates.
- **Interpretation:** **Demolitions tied to bylaw enforcement and energy regulations**.

---

##### Cluster 7 — Deconstruction with Recycling Receipts
- **Size / Share:** 315 permits (5.5%)
- **Avg Project Value:** **$43,507**
- **Top Tokens:** deconstruction, recycle, receipt, confirm, disposal
- **Features:** Detached houses with detailed recycling/disposal compliance.
- **Interpretation:** **Recycling-intensive demolitions** with receipts and confirmation steps.

---

##### Cluster 8 — Single Detached House Demolitions
- **Size / Share:** 190 permits (3.3%)
- **Avg Project Value:** **$22,493**
- **Top Tokens:** detach, single detach, house, low demolish
- **Features:** Nearly all single-detached houses.
- **Interpretation:** **Straightforward demolitions** of single houses, modest costs.

---

##### Cluster 9 — Large Multi-Dwelling Demolitions
- **Size / Share:** 3 permits (0.1%)
- **Avg Project Value:** **$4,648,644**
- **Top Tokens:** demolition, multiple, sign, related, construction
- **Features:** Multiple dwellings, very high average value.
- **Interpretation:** **Rare but very large-scale demolitions** (e.g. apartment blocks).

---


#### Second Optimal (with 5 clusters)

In [91]:
# Run K-Means pipeline with optimal parameters (by sil. score)

demos_kmeans_fitted_01 = run_kmeans_pipeline(cluster_demos_df, n_clusters=5, svd_n_components=8, tfidf_max_features=450)

K-Means pipeline fitted in 0.31 minutes.


In [92]:
# Run full evaluator

full_evaluator(demos_kmeans_fitted_01, cluster_demos_df)



Extracting all tokens from tf-idf vectorizer...


Number of tokens: 408


List of 100 random tokens:

 ['merit', 'reuse', 'contain', 'date', 'right', 'material deconstruction', 'recycle rezoning', 'demolition material', 'engineer', 'suite building', 'waste contractor', 'commercial', 'call', 'great', 'declaration', 'build excavation', 'recycle', 'commencement demolition', 'detach garage', 'shore separate', 'acceptable form', 'mixed use', 'house demolition', 'grade excavation', 'excavating', 'subject', 'build building', 'recycling tonne', 'online', 'grade', 'document', 'report review', 'facility receipt', 'inspection online', 'deconstruct exist', 'family construct', 'acceptable', 'service', 'retail', 'receipt deconstruction', 'garage', 'excavate', 'finalized commencement', 'recycle percent', 'mean green', 'building canadian', 'building excavate', 'demolition request', 'online account', 'attach', 'development', 'excavation', 'contractor update', 'recycle reuse', 'green demo', 'letter', 

Inertia Score (lower better)               30659.287894
Silhouette Score (higher better)               0.429282
Calinski Harabasz Index (higher better)     1090.226122
Davies Bouldin Index (lower better)            1.165817
dtype: float64



Computing feature averages by cluster...


Our clusters have the following feature means:


   cluster  size  share project_value_mean  duplex_w_secondary_suite  \
0        0  1449  25.4%            $17,019                  0.028295   
1        1   517   9.0%            $16,881                  0.015474   
2        2  2342  41.0%            $41,001                  0.008967   
3        3   913  16.0%            $17,336                  0.031763   
4        4   493   8.6%            $17,545                  0.022312   

   laneway_house    duplex  multiple_conversion_dwelling  dwelling_unit  \
0       0.000690  0.043478                      0.000000       0.007591   
1       0.000000  0.034816                      0.000000       0.021277   
2       0.005124  0.039710                      0.006405       0.021776   
3       0.000000  0.089814                      0.000000       0.004381   
4       0.000000  0.054767                      0.000000       0.002028   

   multiple_dwelling  



=== Step runtimes (minutes) ===
- Extracting all tokens: 0.00 min
- Top tokens per SVD component: 0.00 min
- Overall metrics: 0.34 min
- Cluster averages: 0.34 min
- Top tokens per cluster: 0.64 min
- 3D visualization: 0.64 min

Total runtime: 1.96 minutes.


#### K-Means Clustering Results — Demolition / Deconstruction Permits (5 Clusters)

Below is a summary of the 5 clusters, their defining characteristics, and interpretations.

---

##### Cluster 0 — Standard Family House Demolitions
- **Size / Share:** 1,449 permits (25.4%)
- **Avg Project Value:** **$17,019**
- **Top Tokens:** family building, low demolish, construction
- **Features:** Majority single-detached houses, some duplexes with suites.
- **Interpretation:** **Typical low-value demolitions** of family houses.

---

##### Cluster 1 — Green & Recycling-Oriented Demolitions
- **Size / Share:** 517 permits (9.0%)
- **Avg Project Value:** **$16,881**
- **Top Tokens:** green, family building, recycling, deconstruction
- **Features:** Detached houses with emphasis on green compliance and recycling.
- **Interpretation:** **Eco-focused demolitions** with sustainability and recycling requirements.

---

##### Cluster 2 — Higher-Value / Large-Scale Demolitions
- **Size / Share:** 2,342 permits (41.0%)
- **Avg Project Value:** **$41,001**
- **Top Tokens:** deconstruction, demolition, recycle, grade, advance, call, hour
- **Features:** Mix of detached and multiple dwellings; noticeably higher project values.
- **Interpretation:** **Large-scale or complex demolitions**, often requiring scheduling, grading, and contractor oversight.

---

##### Cluster 3 — Hazardous / Non-Hazardous Waste Focus
- **Size / Share:** 913 permits (16.0%)
- **Avg Project Value:** **$17,336**
- **Top Tokens:** construction, hazardous, non hazardous, waste, recycle
- **Features:** Detached/duplex projects tied to waste handling and hazardous materials.
- **Interpretation:** **Demolitions with hazardous material management**, environmental safety emphasis.

---

##### Cluster 4 — Deconstruction & Recycling Emphasis
- **Size / Share:** 493 permits (8.6%)
- **Avg Project Value:** **$17,545**
- **Top Tokens:** deconstruction, mean deconstruction, recycle, green
- **Features:** Detached houses, strong focus on deconstruction and salvage.
- **Interpretation:** **Deconstruction-oriented projects** aimed at recycling and material recovery.

---


## Hierarchical (Agglomerative) Clustering

### Agglomerative Clustering Pipeline Set up

In [93]:
# Define agglomerative clustering pipeline

agglom_pipe = Pipeline(steps=[
    ("prep", preprocess),
    # Toggle this to Normalizer() when using cosine metrics; keep as passthrough for Euclidean/Ward.
    ("norm", "passthrough"),
    ("cluster", AgglomerativeClustering(
        n_clusters=10,              # will be tuned
        linkage="ward",             # default; will be tuned
        metric="euclidean",         # ignored for ward; will be tuned for non-ward
        compute_distances=True      # enables dendrogram plotting later
    ))
])

In [101]:
# Function to run full agglomerative clustering pipeline

def run_agg_pipeline(
    df,
    n_clusters=9,
    svd_n_components=4,
    tfidf_max_features=100,
    metric="euclidean",
    distance_threshold=None,
    label_col="agglom_label"):
    
    """
    Run agglom. pipeline with chosen parameters
    """

    start = time.time()
    
    # Update parameters inside the pipeline
    agglom_pipe.set_params(
        cluster__n_clusters=n_clusters,
        cluster__metric=metric,
        cluster__distance_threshold=distance_threshold,
        prep__text__svd__n_components=svd_n_components,
        prep__text__tfidf__max_features=tfidf_max_features
    )

    
    # Fit the pipeline
    fitted = agglom_pipe.fit(df)

    end = time.time()

    runtime = (end - start)/60

    print(f"Agglomerative clustering pipeline fitted in {runtime:.2f} minutes.")
    
    return fitted

### Optimal Clusterings: Builds

In [166]:
# Drop extreme outliers

print(f"Number of records in our model-ready builds dataframe is {len(cluster_builds_df)}.")

cluster_builds_df = cluster_builds_df[cluster_builds_df['project_value']<500000000]

print(f"After dropping extreme outliers, number of records in our model-ready builds dataframe is {len(cluster_builds_df)}.")

Number of records in our model-ready builds dataframe is 9464.
After dropping extreme outliers, number of records in our model-ready builds dataframe is 9462.


In [167]:
# Run agglomerative clustering pipeline with optimal parameters (by sil. score)

builds_agg_fitted_00 = run_agg_pipeline(cluster_builds_df, metric="euclidean", n_clusters=8, svd_n_components=4, tfidf_max_features=100)

Agglomerative clustering pipeline fitted in 2.27 minutes.


In [168]:
# Get 3D embedding of cluster results

builds_clusters_embedded = cluster_embedding_3d(builds_agg_fitted_00, cluster_builds_df)

In [171]:
builds_clusters_embedded.to_csv(r"C:\Users\emshe\Desktop\BRAINSTATION\CAPSTONE\GIT_REPO\DEMO\data\vis_clusters_builds.csv")

In [169]:
# Plot 3D cluster results

plot_clusters_3d(builds_agg_fitted_00, cluster_builds_df)

In [125]:
# Run full evaluator

full_evaluator(builds_agg_fitted_00, cluster_builds_df)



Extracting all tokens from tf-idf vectorizer...


Number of tokens: 100


List of 100 random tokens:

 ['assign', 'pad', 'projection', 'inspection', 'sprinklered', 'secondary', 'entry', 'construct family', 'compliance', 'number post', 'family', 'entire building', 'build law', 'house', 'soffit expose', 'unit', 'bylaw', 'suite', 'energy bylaw', 'expose', 'parking pad', 'number', 'time build', 'access maintain', 'applicable', 'post suite', 'expose building', 'suite entry', 'detach accessory', 'parking', 'suite number', 'maintain', 'property line', 'title', 'suite locate', 'space have', 'assign address', 'address', 'build visible', 'provide park', 'line', 'garage', 'cellar', 'floor', 'building face', 'access building', 'roof soffit', 'building sprinklered', 'law', 'property', 'family secondary', 'park', 'number assign', 'attach garage', 'build', 'basement', 'secondary suite', 'project property', 'open parking', 'time', 'principal', 'entire', 'construct laneway', 'open', 'accessory buildi

Inertia Score (lower better)                       NaN
Silhouette Score (higher better)              0.376144
Calinski Harabasz Index (higher better)    3765.381617
Davies Bouldin Index (lower better)           0.972425
dtype: float64



Computing feature averages by cluster...


Our clusters have the following feature means:


   cluster  size  share project_value_mean  duplex_w_secondary_suite  \
0        0  1924  20.3%           $657,475                  0.042100   
1        1    47   0.5%       $107,774,490                  0.000000   
2        2  1690  17.9%         $1,627,255                  0.004734   
3        3  1814  19.2%           $239,726                  0.000000   
4        4  1078  11.4%           $244,457                  0.000000   
5        5  1468  15.5%           $886,224                  0.213896   
6        6  1213  12.8%           $942,246                  0.130256   
7        7   228   2.4%        $27,716,589                  0.000000   

   laneway_house    duplex  multiple_conversion_dwelling  dwelling_unit  \
0       0.446466  0.070686                      0.000000       0.003638   
1       0.000000  0.000000                      0.000000       0.021277   
2       0.007101  0.027219      



=== Step runtimes (minutes) ===
- Extracting all tokens: 0.00 min
- Top tokens per SVD component: 0.00 min
- Overall metrics: 1.82 min
- Cluster averages: 1.82 min
- Top tokens per cluster: 3.64 min
- 3D visualization: 3.46 min

Total runtime: 10.75 minutes.


#### Agglomerative Clustering Results — New Building Permits

Below is a summary of the 8 clusters, their defining characteristics, and interpretations.

---

##### Cluster 0 — Small Detached & Duplex Mix
- **Size / Share:** 1,924 permits (20.3%)
- **Avg Project Value:** **$657,475**
- **Top Tokens:** house, face, park, expose, sprinklered, garage
- **Features:** Mix of single-detached homes and duplexes, with some laneway houses.
- **Interpretation:** **Moderate-value detached/duplex housing**, often with fire-safety and exterior features.

---

##### Cluster 1 — Large Multi-Dwelling Projects
- **Size / Share:** 47 permits (0.5%)
- **Avg Project Value:** **$107,774,490**
- **Top Tokens:** unit, park, build, suite, floor, parking
- **Features:** Almost entirely multiple dwellings, very high value.
- **Interpretation:** **Major residential complexes** (condos/high-rises), extreme outliers in value.

---

##### Cluster 2 — Mid-Value Family Housing
- **Size / Share:** 1,690 permits (17.9%)
- **Avg Project Value:** **$1,627,255**
- **Top Tokens:** unit, park, family, garage, cellar
- **Features:** Mix of single-detached and multiple dwellings, with family-oriented tokens.
- **Interpretation:** **Mid- to high-value detached or small multi-unit family housing.**

---

##### Cluster 3 — Laneway Houses (Lower Value)
- **Size / Share:** 1,814 permits (19.2%)
- **Avg Project Value:** **$239,726**
- **Top Tokens:** house, laneway, laneway house, law, construct
- **Features:** Nearly all laneway houses, very low project values.
- **Interpretation:** **Small-scale laneway construction**, low cost per project.

---

##### Cluster 4 — Numbered Suites & Laneways
- **Size / Share:** 1,078 permits (11.4%)
- **Avg Project Value:** **$244,457**
- **Top Tokens:** number, suite, build, house, principal, laneway
- **Features:** Laneway and small detached projects with heavy suite/number references.
- **Interpretation:** **Small dwellings with suites or laneway connections**, low-mid value.

---

##### Cluster 5 — Duplexes with Secondary Suites
- **Size / Share:** 1,468 permits (15.5%)
- **Avg Project Value:** **$886,224**
- **Top Tokens:** suite, number, build, post, floor, basement
- **Features:** High prevalence of duplexes and detached houses with secondary suites.
- **Interpretation:** **Moderate-value duplex housing** with many suites and basement units.

---

##### Cluster 6 — Detached with Suites
- **Size / Share:** 1,213 permits (12.8%)
- **Avg Project Value:** **$942,246**
- **Top Tokens:** locate, suite, unit, secondary, basement, address
- **Features:** Detached homes with a high share of secondary suites.
- **Interpretation:** **Detached housing stock with basement/secondary suites**, solid mid-value.

---

##### Cluster 7 — Large Multi-Unit Midrise Projects
- **Size / Share:** 228 permits (2.4%)
- **Avg Project Value:** **$27,716,589**
- **Top Tokens:** unit, park, build, floor, parking, garage
- **Features:** Predominantly multiple dwellings, high project value.
- **Interpretation:** **Large multi-unit midrise developments**, substantial project costs.

---


In [127]:
# Drop extreme outliers from original builds dataframe

print(f"Number of records in our builds dataframe is {len(builds_df)}.")

builds_df = builds_df[builds_df['project_value']<500000000]

print(f"After dropping extreme outliers, number of records in our model-ready builds dataframe is {len(builds_df)}.")

Number of records in our builds dataframe is 9462.
After dropping extreme outliers, number of records in our model-ready builds dataframe is 9462.


In [128]:
# Add cluster labels

builds_clustered = builds_df.copy()
builds_clustered['cluster'] = builds_agg_fitted_00.named_steps["cluster"].labels_

examine_df('clustered builds', builds_clustered)



Number of records in the clustered builds is: 9462



Number of features in the clustered builds is: 17

The columns in the clustered builds are: Index(['issue_date', 'project_description', 'geom', 'project_value', 'nbhd',
       'zone', 'duplex_w_secondary_suite', 'laneway_house', 'duplex',
       'multiple_conversion_dwelling', 'dwelling_unit', 'multiple_dwelling',
       'single_detached_house', 'single_detached_house_w_sec_suite',
       'permit_category_new_build_low_density_housing',
       'permit_category_new_build_standalone_laneway', 'cluster'],
      dtype='object')


 Other info about clustered builds:

<class 'pandas.core.frame.DataFrame'>
Index: 9462 entries, 0 to 9463
Data columns (total 17 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   issue_date                                     9462 non-null   object 
 1   project_description                     

None


 Basic statistical info about clustered builds:



,project_value,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,permit_category_new_build_low_density_housing,permit_category_new_build_standalone_laneway,cluster
count,9.462000e+03,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000
mean,1.959642e+06,0.059290,0.392517,0.096703,0.001163,0.004122,0.063940,0.138026,0.225639,0.692560,0.212957,3.106637
std,9.530098e+06,0.236179,0.488337,0.295568,0.034078,0.064072,0.244659,0.344945,0.418025,0.461458,0.409419,2.071581
min,1.019777e+03,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.387938e+05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000
50%,7.068476e+05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,3.000000
75%,9.782168e+05,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,5.000000
max,3.498087e+08,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,7.000000




Sample of records in the clustered builds:


,issue_date,project_description,geom,project_value,nbhd,zone,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,permit_category_new_build_low_density_housing,permit_category_new_build_standalone_laneway,cluster
0,2017-04-12,Low Density Housing - New Building - To constr...,POINT (-123.0438851 49.2545202),2.501153e+05,Renfrew,Mount Pleasant/Renfrew Heights,0,0,0,0,0,0,0,1,1,0,3
1,2017-09-28,Low Density Housing - New Building - To constr...,POINT (-123.0846679 49.2365573),1.838661e+05,Sunset,Southeast Vancouver,0,1,0,0,0,0,0,0,0,1,3
2,2017-11-27,Low Density Housing - New Building - To constr...,POINT (-123.0342549 49.254328),8.857129e+05,Renfrew,Mount Pleasant/Renfrew Heights,0,0,0,0,0,0,1,0,1,0,2
3,2024-12-06,Low Density Housing - New Building - To constr...,POINT (-123.0629329 49.2168193),9.618750e+05,Fraser View/Killarny,Southeast Vancouver,1,0,0,0,0,0,0,0,1,0,5
4,2024-05-03,Low Density Housing - New Building - To constr...,POINT (-123.051063 49.2669312),1.056125e+06,Hastings/Sunrise/Grandview/Woodlands,East Hastings,1,0,0,0,0,0,0,0,1,0,6


In [129]:
# Save new building permits with cluster labels as a csv

builds_clustered.to_csv(f"{PATH}/CLUSTERED/builds_clustered.csv", index=False, encoding="utf-8")

### Optimal Clusterings: Renos

In [172]:
# Run K-Means pipeline with optimal parameters  (by sil. score)

renos_agg_fitted_00 = run_agg_pipeline(cluster_renos_df, metric="euclidean", n_clusters=6, svd_n_components=3, tfidf_max_features=300)

Agglomerative clustering pipeline fitted in 1.03 minutes.


In [173]:
# Get 3D embedding of cluster results

renos_clusters_embedded = cluster_embedding_3d(renos_agg_fitted_00, cluster_renos_df)

In [174]:
renos_clusters_embedded.to_csv(r"C:\Users\emshe\Desktop\BRAINSTATION\CAPSTONE\GIT_REPO\DEMO\data\vis_clusters_renos.csv")

In [133]:
# Run full evaluator

full_evaluator(renos_agg_fitted_00, cluster_renos_df)



Extracting all tokens from tf-idf vectorizer...


Number of tokens: 300


List of 100 random tokens:

 ['hardwired volt', 'assurance', 'wall kitchen', 'new window', 'silence', 'detector', 'millwork', 'install', 'build scope', 'support document', 'floor multiple', 'improvement exist', 'replace', 'exist multiple', 'floor', 'plumbing', 'inside', 'paint', 'installation', 'building scope', 'inspection', 'upgrade', 'alteration improvement', 'insulation', 'stage specifically', 'washer dryer', 'alarm', 'electrical', 'area', 'mixed', 'alteration remove', 'energy upgrade', 'responsibility', 'dryer', 'construction', 'building field', 'address', 'alteration repair', 'check', 'floor alteration', 'service', 'volt smoke', 'light fixture', 'mixed use', 'kitchen bathroom', 'bedroom', 'scope', 'currently', 'structural', 'light', 'partition wall', 'applicable', 'structural engineer', 'checklist', 'envelope', 'stratum title', 'use exist', 'tile', 'address post', 'time', 'silence sleep', 'efficient', 'lo

Inertia Score (lower better)                       NaN
Silhouette Score (higher better)              0.352394
Calinski Harabasz Index (higher better)    3076.127147
Davies Bouldin Index (lower better)           0.974570
dtype: float64



Computing feature averages by cluster...


Our clusters have the following feature means:


   cluster  size  share project_value_mean  duplex_w_secondary_suite  \
0        0  3652  46.4%            $68,301                  0.000000   
1        1  2340  29.7%            $54,300                  0.002991   
2        2   402   5.1%           $481,548                  0.000000   
3        3   949  12.1%            $59,653                  0.001054   
4        4    38   0.5%         $2,478,865                  0.000000   
5        5   494   6.3%            $60,883                  0.000000   

   laneway_house    duplex  multiple_conversion_dwelling  dwelling_unit  \
0       0.000000  0.029573                      0.019168       0.047097   
1       0.012821  0.037607                      0.022650       0.029487   
2       0.000000  0.017413                      0.009950       0.052239   
3       0.000000  0.000000                      0.016860       0.002107   
4       0.000000  0.000000



=== Step runtimes (minutes) ===
- Extracting all tokens: 0.00 min
- Top tokens per SVD component: 0.00 min
- Overall metrics: 0.90 min
- Cluster averages: 0.89 min
- Top tokens per cluster: 1.80 min
- 3D visualization: 1.78 min

Total runtime: 5.37 minutes.


#### Agglomerative Clustering Results — Renovation Permits

Below is a summary of the 6 clusters, their defining characteristics, and interpretations.

---

##### Cluster 0 — Multi-Unit Interior Renovations
- **Size / Share:** 3,652 permits (46.4%)
- **Avg Project Value:** **$68,301**
- **Top Tokens:** floor, unit, building, multiple, kitchen, wall
- **Features:** Predominantly multiple dwellings; lots of interior improvements (floors, kitchens, bathrooms).
- **Interpretation:** **Routine low-cost renovations** in multi-unit residential buildings.

---

##### Cluster 1 — Single-Detached Home Renovations
- **Size / Share:** 2,340 permits (29.7%)
- **Avg Project Value:** **$54,300**
- **Top Tokens:** new, family, floor, remove, building, interior, door
- **Features:** Mostly single-detached houses; focus on general interior/exterior remodeling.
- **Interpretation:** **Mainstream single-family home renovations**, low- to mid-value.

---

##### Cluster 2 — Higher-Value Renovations (Windows, Roofs, Exteriors)
- **Size / Share:** 402 permits (5.1%)
- **Avg Project Value:** **$481,548**
- **Top Tokens:** new, building, floor, interior, window, door, repair, upgrade
- **Features:** Detached and multiple dwellings; scope includes exteriors, windows, and major repairs.
- **Interpretation:** **High-value renovation projects**, more comprehensive in scope.

---

##### Cluster 3 — Secondary Suite Renovations
- **Size / Share:** 949 permits (12.1%)
- **Avg Project Value:** **$59,653**
- **Top Tokens:** suite, secondary, secondary suite, family, access, law
- **Features:** Almost all projects involve adding or upgrading secondary suites.
- **Interpretation:** **Suite-focused renovations**, moderate value.

---

##### Cluster 4 — Large-Scale Structural Repairs
- **Size / Share:** 38 permits (0.5%)
- **Avg Project Value:** **$2,478,865**
- **Top Tokens:** new, building, roof, replace, repair, multiple, tree
- **Features:** Mix of multiple dwellings; significant structural scope (roof, repairs).
- **Interpretation:** **Rare but very large renovation projects**, extremely high values.

---

##### Cluster 5 — Multi-Unit Alterations
- **Size / Share:** 494 permits (6.3%)
- **Avg Project Value:** **$60,883**
- **Top Tokens:** multiple, building, unit, alteration, improvement
- **Features:** Almost entirely multiple dwellings; direct alterations and improvements.
- **Interpretation:** **Multi-unit building alterations**, mid-value renovation projects.

---


In [134]:
# Add cluster labels

renos_clustered = renos_df.copy()
renos_clustered['cluster'] = renos_agg_fitted_00.named_steps["cluster"].labels_

examine_df('clustered renos_df', renos_clustered)



Number of records in the clustered renos_df is: 7875



Number of features in the clustered renos_df is: 15

The columns in the clustered renos_df are: Index(['issue_date', 'project_description', 'geom', 'project_value', 'nbhd',
       'zone', 'duplex_w_secondary_suite', 'laneway_house', 'duplex',
       'multiple_conversion_dwelling', 'dwelling_unit', 'multiple_dwelling',
       'single_detached_house', 'single_detached_house_w_sec_suite',
       'cluster'],
      dtype='object')


 Other info about clustered renos_df:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7875 entries, 0 to 7874
Data columns (total 15 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   issue_date                         7875 non-null   object 
 1   project_description                7875 non-null   object 
 2   geom                               7875 non-null   object 
 3   project_value                      7875

None


 Basic statistical info about clustered renos_df:



,project_value,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,cluster
count,7.875000e+03,7875.000000,7875.000000,7875.000000,7875.000000,7875.000000,7875.000000,7875.000000,7875.000000,7875.000000
mean,9.536062e+04,0.001016,0.003810,0.026032,0.018159,0.049270,0.501968,0.252698,0.131556,1.093714
std,2.235591e+05,0.031859,0.061608,0.159240,0.133534,0.216445,0.500028,0.434587,0.338028,1.426679
min,5.053050e+02,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.594619e+04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,4.000000e+04,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
75%,9.434814e+04,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000
max,6.000000e+06,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5.000000




Sample of records in the clustered renos_df:


,issue_date,project_description,geom,project_value,nbhd,zone,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,cluster
0,2018-03-06,Field Review - Addition / Alteration - #308\n\...,POINT (-123.156148 49.2705801),26818.668707,Kitsilano/Point Grey North,Kitsilano/Point Grey,0,0,0,0,0,1,0,0,0
1,2021-03-11,Field Review - Addition / Alteration - Exterio...,POINT (-123.0571046 49.2611421),221995.630007,Renfrew,Mount Pleasant/Renfrew Heights,0,0,0,0,0,0,1,0,1
2,2022-03-01,Field Review - Addition / Alteration - Interio...,POINT (-123.0894788 49.2616599),119794.941900,Mount Pleasant,Mount Pleasant/Renfrew Heights,0,0,0,1,0,0,0,0,0
3,2017-03-20,Field Review - Addition / Alteration - Exterio...,POINT (-123.0742693 49.2220461),60961.195294,Sunset,Southeast Vancouver,0,0,0,0,0,0,0,1,3
4,2021-10-28,Field Review - Addition / Alteration - Exterio...,POINT (-123.0263275 49.2088643),58828.841952,Fraser View/Killarny,Southeast Vancouver,0,0,0,0,0,1,0,0,0


In [135]:
# Save new building permits with cluster labels as a csv

renos_clustered.to_csv(f"{PATH}/CLUSTERED/renos_clustered.csv", index=False, encoding="utf-8")

### Optimal Clusterings: Demos

#### First Optimal (with 8 Clusters)

In [106]:
# Run K-Means pipeline with optimal parameters (by sil. score)

demos_agg_fitted_00 = run_agg_pipeline(cluster_demos_df, n_clusters=8, svd_n_components=2, tfidf_max_features=300)

Agglomerative clustering pipeline fitted in 0.36 minutes.


In [107]:
# Run full evaluator

full_evaluator(demos_agg_fitted_00, cluster_demos_df)



Extracting all tokens from tf-idf vectorizer...


Number of tokens: 300


List of 100 random tokens:

 ['recycling tonne', 'related', 'demolish family', 'build demolition', 'construction waste', 'tonne', 'family grade', 'confirm', 'sign', 'build', 'development', 'deconstruction build', 'building grade', 'exist building', 'grade', 'contain unit', 'recycle demolition', 'waste', 'contain', 'excavation soil', 'green suspend', 'acceptable', 'material update', 'zone', 'exist detach', 'green recycling', 'advance demolition', 'construction', 'material deconstruction', 'deconstruction building', 'demolition', 'subdivision order', 'receipt attach', 'order', 'family relate', 'update', 'form', 'recycling material', 'building sign', 'recycling sign', 'recycle deconstruction', 'attach', 'disposal', 'demolition hour', 'salvage abatement', 'pending', 'facility receipt', 'excavate', 'percent non', 'recycle construction', 'character', 'family building', 'recycle material', 'copy', 'excavation', 'famil

Inertia Score (lower better)                       NaN
Silhouette Score (higher better)              0.589485
Calinski Harabasz Index (higher better)    6079.450004
Davies Bouldin Index (lower better)           0.754736
dtype: float64



Computing feature averages by cluster...


Our clusters have the following feature means:


   cluster  size  share project_value_mean  duplex_w_secondary_suite  \
0        0  1905  33.3%            $24,662                  0.000000   
1        1   575  10.1%            $17,081                  0.081739   
2        2     3   0.1%         $4,648,644                  0.000000   
3        3   530   9.3%            $34,600                  0.041509   
4        4    30   0.5%           $648,754                  0.000000   
5        5   884  15.5%            $16,921                  0.000000   
6        6  1209  21.2%            $16,925                  0.000000   
7        7   578  10.1%            $17,164                  0.070934   

   laneway_house    duplex  multiple_conversion_dwelling  dwelling_unit  \
0       0.000000  0.000000                      0.000000       0.000000   
1       0.000000  0.222609                      0.000000       0.052174   
2       0.000000  0.000000      



=== Step runtimes (minutes) ===
- Extracting all tokens: 0.00 min
- Top tokens per SVD component: 0.00 min
- Overall metrics: 0.37 min
- Cluster averages: 0.35 min
- Top tokens per cluster: 0.68 min
- 3D visualization: 0.69 min

Total runtime: 2.09 minutes.


#### Agglomerative Clustering Results — Demolition / Deconstruction Permits

Below is a summary of the 8 clusters, their defining characteristics, and interpretations.

---

##### Cluster 0 — Standard Single-Family Demolitions
- **Size / Share:** 1,905 permits (33.3%)
- **Avg Project Value:** **$24,662**
- **Top Tokens:** deconstruction, recycle, demolition, family building, low demolish
- **Features:** All single-detached houses; frequent references to recycling and receipts.
- **Interpretation:** **Typical demolitions of single-family homes**, modest project values with recycling compliance.

---

##### Cluster 1 — Green/Mean Deconstruction
- **Size / Share:** 575 permits (10.1%)
- **Avg Project Value:** **$17,081**
- **Top Tokens:** mean, building mean, deconstruction, family building, low demolish
- **Features:** Many duplexes; “mean deconstruction” and green subject/recycling references.
- **Interpretation:** **Green-focused deconstruction projects**, eco-compliance emphasis.

---

##### Cluster 2 — Large Multi-Dwelling Demolitions
- **Size / Share:** 3 permits (0.1%)
- **Avg Project Value:** **$4,648,644**
- **Top Tokens:** demolition, new, build, basement, mixed, law
- **Features:** Mostly multiple dwellings, extremely high-value.
- **Interpretation:** **Rare, very large-scale demolitions**, e.g. apartment blocks.

---

##### Cluster 3 — Demolitions with Soil/Grading Issues
- **Size / Share:** 530 permits (9.3%)
- **Avg Project Value:** **$34,600**
- **Top Tokens:** demolition, deconstruction, low demolish, soil, grade
- **Features:** Duplex-heavy projects; frequent mentions of grading, soil, and contractor calls.
- **Interpretation:** **Medium-value demolitions** involving soil/grade management.

---

##### Cluster 4 — Scheduled/Advance Demolitions
- **Size / Share:** 30 permits (0.5%)
- **Avg Project Value:** **$648,754**
- **Top Tokens:** demolition, call, hour advance, outside, grade
- **Features:** Duplex/multiple dwellings; higher values, explicit scheduling tokens.
- **Interpretation:** **Time-sensitive demolition projects** with structured scheduling, higher costs.

---

##### Cluster 5 — Low-Cost Family House Demolitions
- **Size / Share:** 884 permits (15.5%)
- **Avg Project Value:** **$16,921**
- **Top Tokens:** family building, low demolish, recycling
- **Features:** Entirely single-detached houses, no suites or multiples.
- **Interpretation:** **Basic demolitions of family homes**, very low value.

---

##### Cluster 6 — Eco/Green Deconstruction
- **Size / Share:** 1,209 permits (21.2%)
- **Avg Project Value:** **$16,925**
- **Top Tokens:** deconstruction, family building, mean deconstruction, green, recycle
- **Features:** Detached housing; strong focus on “green” deconstruction and recycling.
- **Interpretation:** **Eco-oriented deconstruction cluster**, low-value but environmentally framed.

---

##### Cluster 7 — Duplex/Secondary Suite Demolitions
- **Size / Share:** 578 permits (10.1%)
- **Avg Project Value:** **$17,164**
- **Top Tokens:** family building, low demolish, subject, deposit, detach
- **Features:** High rate of duplexes and secondary suites; compliance-heavy language.
- **Interpretation:** **Demolitions of duplexes or secondary-suite dwellings**, modest project values.

---


#### Second Optimal (with 4 clusters)

In [175]:
# Run K-Means pipeline with optimal parameters (by sil. score)

demos_agg_fitted_01 = run_agg_pipeline(cluster_demos_df, n_clusters=4, svd_n_components=2, tfidf_max_features=300)

Agglomerative clustering pipeline fitted in 0.40 minutes.


In [176]:
# Get 3D embedding of cluster results

demos_clusters_embedded = cluster_embedding_3d(demos_agg_fitted_00, cluster_demos_df)

In [177]:
demos_clusters_embedded.to_csv(r"C:\Users\emshe\Desktop\BRAINSTATION\CAPSTONE\GIT_REPO\DEMO\data\vis_clusters_demos.csv")

In [138]:
# Run full evaluator

full_evaluator(demos_agg_fitted_01, cluster_demos_df)



Extracting all tokens from tf-idf vectorizer...


Number of tokens: 300


List of 100 random tokens:

 ['contain unit', 'subdivision', 'demolish single', 'recycle confirm', 'construction', 'detach', 'form', 'demolition relate', 'percent non', 'commencement', 'hazardous', 'family grade', 'law', 'demolition green', 'tonne', 'issuable', 'house demolition', 'low contractor', 'family relate', 'material deconstruction', 'rental building', 'building relate', 'assess', 'receipt attach', 'abatement', 'excavation soil', 'sign', 'exist mixed', 'barrier', 'update energy', 'uploaded', 'outside shore', 'detach house', 'advance demolition', 'build contractor', 'family recycle', 'document', 'building update', 'related salvage', 'subdivision order', 'hour', 'receipt document', 'relate', 'building pre', 'residential rental', 'contractor contractor', 'secondary suite', 'compliancereport deconstruction', 'update', 'energy', 'demolition call', 'building assess', 'waste', 'excavate subject', 'copy recycle

Inertia Score (lower better)                       NaN
Silhouette Score (higher better)              0.547196
Calinski Harabasz Index (higher better)    5845.553982
Davies Bouldin Index (lower better)           0.641214
dtype: float64



Computing feature averages by cluster...


Our clusters have the following feature means:


   cluster  size  share project_value_mean  duplex_w_secondary_suite  \
0        0  2465  43.1%            $34,394                  0.008925   
1        1  1462  25.6%            $17,017                  0.028044   
2        2     3   0.1%         $4,648,644                  0.000000   
3        3  1784  31.2%            $16,975                  0.026345   

   laneway_house    duplex  multiple_conversion_dwelling  dwelling_unit  \
0       0.004868  0.036917                      0.006085       0.015010   
1       0.000684  0.043776                      0.000000       0.007524   
2       0.000000  0.000000                      0.000000       0.000000   
3       0.000000  0.071749                      0.000000       0.016816   

   multiple_dwelling  single_detached_house  \
0           0.036917               0.774442   
1           0.000000               0.604651   
2           0.666667        



=== Step runtimes (minutes) ===
- Extracting all tokens: 0.00 min
- Top tokens per SVD component: 0.00 min
- Overall metrics: 0.35 min
- Cluster averages: 0.35 min
- Top tokens per cluster: 0.66 min
- 3D visualization: 0.68 min

Total runtime: 2.04 minutes.


#### Agglomerative Clustering Results — Demolition / Deconstruction Permits (4 Clusters)

Below is a summary of the 4 clusters, their defining characteristics, and interpretations.

---

##### Cluster 0 — Deconstruction & Recycling (Higher Value)
- **Size / Share:** 2,465 permits (43.1%)
- **Avg Project Value:** **$34,394**
- **Top Tokens:** deconstruction, demolition, recycle, low demolish, green, grade
- **Features:** Mix of single-detached and some multiple dwellings; emphasis on recycling and deconstruction.
- **Interpretation:** **Moderately higher-value demolitions** with sustainability and material recovery focus.

---

##### Cluster 1 — Standard Family House Demolitions
- **Size / Share:** 1,462 permits (25.6%)
- **Avg Project Value:** **$17,017**
- **Top Tokens:** family building, low demolish, recycling
- **Features:** Predominantly single-detached houses, some duplexes with suites.
- **Interpretation:** **Typical demolitions of small family homes**, low-value and straightforward.

---

##### Cluster 2 — Large Multi-Dwelling Demolitions
- **Size / Share:** 3 permits (0.1%)
- **Avg Project Value:** **$4,648,644**
- **Top Tokens:** demolition, new, build, basement, mixed, law
- **Features:** Multiple dwellings only; extremely high-value.
- **Interpretation:** **Rare, large-scale demolition projects**, e.g. apartments or mixed-use complexes.

---

##### Cluster 3 — Green/Mean Deconstruction
- **Size / Share:** 1,784 permits (31.2%)
- **Avg Project Value:** **$16,975**
- **Top Tokens:** mean, deconstruction, building mean, family building, low demolish, green
- **Features:** Detached houses and duplexes; heavy references to “mean deconstruction,” recycling, and green compliance.
- **Interpretation:** **Eco-focused deconstruction projects**, standard single-house demolitions framed as green/eco-friendly.

---


In [139]:
# Add cluster labels

demos_clustered = demos_df.copy()
demos_clustered['cluster'] = demos_agg_fitted_01.named_steps["cluster"].labels_

examine_df('clustered demos_df', demos_clustered)



Number of records in the clustered demos_df is: 5714



Number of features in the clustered demos_df is: 15

The columns in the clustered demos_df are: Index(['issue_date', 'project_description', 'geom', 'project_value', 'nbhd',
       'zone', 'duplex_w_secondary_suite', 'laneway_house', 'duplex',
       'multiple_conversion_dwelling', 'dwelling_unit', 'multiple_dwelling',
       'single_detached_house', 'single_detached_house_w_sec_suite',
       'cluster'],
      dtype='object')


 Other info about clustered demos_df:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5714 entries, 0 to 5713
Data columns (total 15 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   issue_date                         5714 non-null   object 
 1   project_description                5714 non-null   object 
 2   geom                               5714 non-null   object 
 3   project_value                      5714

None


 Basic statistical info about clustered demos_df:



,project_value,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,cluster
count,5.714000e+03,5714.000000,5714.000000,5714.000000,5714.000000,5714.000000,5714.000000,5714.000000,5714.000000,5714.000000
mean,2.693202e+04,0.019251,0.002275,0.049527,0.002625,0.013651,0.016451,0.700385,0.190060,1.193560
std,1.215656e+05,0.137418,0.047648,0.216986,0.051173,0.116046,0.127213,0.458129,0.392382,1.282032
min,1.109978e+03,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.562543e+04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.708520e+04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000
75%,1.792941e+04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,3.000000
max,6.500000e+06,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.000000




Sample of records in the clustered demos_df:


,issue_date,project_description,geom,project_value,nbhd,zone,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,cluster
0,2022-11-14,Low Density Housing - Demolition / Deconstruct...,POINT (-123.0677731 49.2249324),15625.427204,Fraser View/Killarny,Southeast Vancouver,0,0,0,0,0,0,1,0,3
1,2017-08-23,Field Review - Demolition / Deconstruction - T...,POINT (-123.1211653 49.2588709),75669.289412,South Granville,South Granville/Oak,0,0,0,0,0,1,0,0,0
2,2018-08-03,Low Density Housing - Demolition / Deconstruct...,POINT (-123.0894871 49.239411),17490.436113,Sunset,Southeast Vancouver,0,0,0,0,0,0,0,1,3
3,2024-12-06,Low Density Housing - Demolition / Deconstruct...,POINT (-123.0629329 49.2168193),15000.000000,Fraser View/Killarny,Southeast Vancouver,0,0,0,0,0,0,1,0,1
4,2021-02-11,Low Density Housing - Demolition / Deconstruct...,POINT (-123.0271394 49.2367555),16649.672251,Collingwood,Southeast Vancouver,0,0,0,0,0,0,1,0,3


In [140]:
# Save new building permits with cluster labels as a csv

demos_clustered.to_csv(f"{PATH}/CLUSTERED/demos_clustered.csv", index=False, encoding="utf-8")